## 分组

### 1.分组模式及其对象

#### 1.1 分组的一般模式

##### 练一练：请在learn_pandas数据集上按学校分组统计体重的均值。

In [37]:
import pandas as pd
import numpy as np
from sqlalchemy.util import ellipses_string

df = pd.read_csv("data_base/data/learn_pandas.csv")
df.groupby('School')['Weight'].mean()

School
A    56.442308
B    55.666667
C    54.000000
D    54.223881
Name: Weight, dtype: float64

#### 1.2 分组依据的本质

In [7]:
q25 = df.Weight.quantile(0.25)
q75 = df.Weight.quantile(0.75)
w_dict = {0:"low", 1:"normal", 2:"high"}
condition = ((df.Weight > q25)*1 + (df.Weight > q75)*1).replace(w_dict)
df.groupby(condition)["Height"].mean()

Weight
high      174.935714
low       155.891071
normal    162.255294
Name: Height, dtype: float64

#### 1.3 groupby对象

In [9]:
gp = df.groupby(['School','Grade'])
gp

In [10]:
# 分组的数量
gp.ngroups

16

In [13]:
res = gp.groups
res.keys()

dict_keys([('A', 'Freshman'), ('A', 'Junior'), ('A', 'Senior'), ('A', 'Sophomore'), ('B', 'Freshman'), ('B', 'Junior'), ('B', 'Senior'), ('B', 'Sophomore'), ('C', 'Freshman'), ('C', 'Junior'), ('C', 'Senior'), ('C', 'Sophomore'), ('D', 'Freshman'), ('D', 'Junior'), ('D', 'Senior'), ('D', 'Sophomore')])

In [15]:
# size()
df.iloc[:5,:5].size
gp.size()

School  Grade    
A       Freshman     13
        Junior       17
        Senior       22
        Sophomore     5
B       Freshman     13
        Junior        8
        Senior        8
        Sophomore     5
C       Freshman      9
        Junior       12
        Senior       11
        Sophomore     8
D       Freshman     17
        Junior       22
        Senior       14
        Sophomore    16
dtype: int64

In [17]:
gp.get_group(('A','Freshman')).iloc[:5,:5]

,School,Grade,Name,Gender,Height
0,A,Freshman,Gaopeng Yang,Female,158.9
6,A,Freshman,Qiang Chu,Female,162.5
10,A,Freshman,Xiaopeng Zhou,Male,174.1
60,A,Freshman,Yanpeng Lv,Male,NaN
114,A,Freshman,Xiaopeng Zhao,Female,161.0


### 2.聚合函数 -> 本质是对Series类型进行聚合

#### 2.1内置聚合函数

In [29]:
# 练一练:在learn_pandas数据集中，Transfer列的元素为“N”时表示该名同学不是转系生，请按照学校和年级两列分组，找出所有不含转系生的组对应的学校和年级。

res = (df.Transfer=="N").groupby([df.School, df.Grade]).all()
res # 等价于res[res == True]

School  Grade    
A       Freshman      True
        Junior        True
        Senior       False
        Sophomore     True
B       Freshman     False
        Junior       False
        Senior       False
        Sophomore    False
C       Freshman      True
        Junior       False
        Senior       False
        Sophomore     True
D       Freshman     False
        Junior       False
        Senior       False
        Sophomore    False
Name: Transfer, dtype: bool

In [8]:
s = pd.Series([True,False,True,False])
s[s]

0    True
2    True
dtype: bool

#### 2.2 agg()函数

In [10]:
# 练一练：请使用传入字典的方法完成与gb.agg(['max', 'min'])等价的聚合任务。

gb = df.groupby('Gender')[['Weight','Height']]
gb.agg({'Weight':['max','min'],'Height':['max','min']})

Weight       Height       
          max   min    max    min
Gender                           
Female   63.0  34.0  170.2  145.4
Male     89.0  51.0  193.9  155.7

In [12]:
gb.describe()

Weight                                                     Height  \
        count       mean       std   min   25%   50%    75%   max  count   
Gender                                                                     
Female  135.0  47.918519  5.405983  34.0  44.0  48.0  52.00  63.0  132.0   
Male     54.0  72.759259  7.772557  51.0  69.0  73.0  78.75  89.0   51.0   

                                                                    
             mean       std    min      25%    50%      75%    max  
Gender                                                              
Female  159.19697  5.053982  145.4  155.675  159.6  162.825  170.2  
Male    173.62549  7.048485  155.7  168.900  173.4  177.150  193.9

In [16]:
# 练一练：在groupby对象上可以使用describe()方法进行统计信息汇总，请同时使用多个聚合函数，完成与该方法相同的功能。

gb.agg(
    ['count','mean','std','min',
     ('25%',lambda x:x.quantile(0.25)), # 聚合结果重命名
     ('50%',lambda x:x.quantile(0.5)),
     ('75%',lambda x:x.quantile(0.75)),
     'max']
)

Weight                                                     Height  \
        count       mean       std   min   25%   50%    75%   max  count   
Gender                                                                     
Female    135  47.918519  5.405983  34.0  44.0  48.0  52.00  63.0    132   
Male       54  72.759259  7.772557  51.0  69.0  73.0  78.75  89.0     51   

                                                                    
             mean       std    min      25%    50%      75%    max  
Gender                                                              
Female  159.19697  5.053982  145.4  155.675  159.6  162.825  170.2  
Male    173.62549  7.048485  155.7  168.900  173.4  177.150  193.9

In [20]:
def my_func(s):
    res = 'High'
    if s.mean() < df[s.name].mean():
        res = 'Low'
    return res
gb.agg(my_func)

,Weight,Height
Gender,,
Female,Low,Low
Male,High,High


### 3.变换和过滤

#### 1.变换函数

In [3]:
import pandas as pd

example = pd.DataFrame({"A":list('abbaab'),"B":[3,5,6,2,1,7]})
example.groupby("A")["B"].cummax() # 先分组计算，后排序。

0    3
1    5
2    6
3    3
4    3
5    7
Name: B, dtype: int64

In [8]:
# 练一练：在groupby对象中，rank()方法也是一个实用的变换函数，请在官方文档中查阅它的功能并给出1个使用的例子。

df = pd.read_csv("data_base/data/learn_pandas.csv")
df["年级内体重排名"] = df.groupby("Grade")["Weight"].rank(ascending=False)
# 体重最重排第一

df.loc[df["年级内体重排名"] < 3,["Grade","Name","Weight","年级内体重排名"]].sort_values(["Grade","年级内体重排名"])

,Grade,Name,Weight,年级内体重排名
38,Freshman,Qiang Han,87.0,1.0
99,Freshman,Changpeng Zhao,83.0,2.5
117,Freshman,Chunli Zhao,83.0,2.5
82,Junior,Changfeng Lv,76.0,1.5
158,Junior,Chengqiang Zhang,76.0,1.5
2,Senior,Mei Sun,89.0,1.0
23,Senior,Qiang Zheng,87.0,2.0
71,Sophomore,Feng Han,82.0,1.0
40,Sophomore,Li Wang,79.0,2.5
48,Sophomore,Mei Xu,79.0,2.5


In [11]:
gb.transform('mean').head()

,Weight,Height
0,47.918519,159.19697
1,72.759259,173.62549
2,72.759259,173.62549
3,47.918519,159.19697
4,72.759259,173.62549


In [12]:
example.groupby("A")["A"].transform(lambda x:0 if x.name == "a"else 1)

0    0
1    1
2    1
3    0
4    0
5    1
Name: A, dtype: int64

#### 2.组索引和过滤

In [13]:
gb.filter(lambda x:x.shape[0] > 100).head()

,Weight,Height
0,46.0,158.9
3,41.0,NaN
5,51.0,158.0
6,52.0,162.5
7,50.0,161.9


In [26]:
#练习：找出所有"没有转系生"的 (School, Grade) 组。(不方便)
res = df.groupby(['School', 'Grade']).filter(
    lambda x: (x['Transfer'] == 'N').all()
)
res.loc[:,['School', 'Grade']].drop_duplicates().sort_values(['School','Grade'])

,School,Grade
0,A,Freshman
31,A,Junior
13,A,Sophomore
15,C,Freshman
3,C,Sophomore


In [30]:
# 练一练：从概念上说，索引功能是组过滤功能的子集，请使用groupby对象上的filter()方法完成loc[...]的功能，这里假设“...”是元素的列表。
df_new = pd.DataFrame({"A": [1, 2, 3, 4, 5, 6]},index=list('abcdef'))

,A
a,1
b,2
c,3
d,4
e,5
f,6


In [31]:
item_list = ["b","d","f"]
df_new.groupby(df.index).filter(lambda x:x.index[0] in item_list)

,A
b,2
d,4
f,6


### 4.跨列分组

#### 1.标量的情况

In [46]:
gb1 = df.groupby(['Gender','Test_Number'])['Height']
gb2 = df.groupby(['Gender','Test_Number'])[['Height','Weight']]
gb1.apply(lambda x: 0)

Gender  Test_Number
Female  1              0
        2              0
        3              0
Male    1              0
        2              0
        3              0
Name: Height, dtype: int64

In [50]:
gb2.apply(lambda x: [0,0])

Gender  Test_Number
Female  1              [0, 0]
        2              [0, 0]
        3              [0, 0]
Male    1              [0, 0]
        2              [0, 0]
        3              [0, 0]
dtype: object

#### 2.Series的情况

In [51]:
gb1.apply(lambda x: pd.Series([0,1],index=['a','b']))

Gender  Test_Number   
Female  1            a    0
                     b    1
        2            a    0
                     b    1
        3            a    0
                     b    1
Male    1            a    0
                     b    1
        2            a    0
                     b    1
        3            a    0
                     b    1
Name: Height, dtype: int64

In [52]:
gb2.apply(lambda x: pd.Series([0,1],index=['a','b']))

a  b
Gender Test_Number      
Female 1            0  1
       2            0  1
       3            0  1
Male   1            0  1
       2            0  1
       3            0  1

#### 3.DataFrame

In [55]:
import numpy as np

temp_df = pd.DataFrame(np.ones((2,2)),
                       index=['a','b'],
                       columns=pd.Index([('w','x'),('y','z')]))
gb1.apply(lambda x:temp_df).head()

w    y
                        x    z
Gender Test_Number            
Female 1           a  1.0  1.0
                   b  1.0  1.0
       2           a  1.0  1.0
                   b  1.0  1.0
       3           a  1.0  1.0

In [56]:
gb2.apply(lambda x:temp_df).head()

w    y
                        x    z
Gender Test_Number            
Female 1           a  1.0  1.0
                   b  1.0  1.0
       2           a  1.0  1.0
                   b  1.0  1.0
       3           a  1.0  1.0